In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from scipy import stats
import plotly.graph_objects as go
import plotly.io as pio


list of TFs enriched in episodic

In [2]:

all_path = "/work/nvme/bhdw/asachan/data_files/firefate/bcell/celloracle_PS_files/Tonsil_all_scores.csv"

all = pd.read_csv(all_path)


In [3]:
all.shape

(107, 16)

In [4]:
# All TFs from dictys hits based on curve shape analysis
dictys_tfs = [
    "BACH2", "BATF3", "BHLHE22", "BHLHE40", "BHLHE41", "E2F4", "E2F6",
    "EGR1", "EGR2", "EGR3", "ETV5", "FOS", "FOSL2", "GLI1", "HES1",
    "HNF1B", "HOMEZ", "HSF5", "IRF8", "KLF9", "LEF1", "MEF2B", "MYC",
    "MYPOP", "NFATC4", "NFIL3", "NR4A1", "PATZ1", "PBX4", "POU2F3",
    "PPARA", "PRDM1", "REL", "RUNX2", "SNAI3", "SOX2", "SOX5", "SREBF1",
    "ST18", "TBX21", "TCF7L2", "TFAP4", "TGIF1", "THRB", "TP73", "VDR",
    "XBP1", "YBX1", "ZBTB17", "ZBTB48", "ZFP3", "ZFP14", "ZNF16",
    "ZNF35", "ZNF202", "ZNF222", "ZNF385D", "ZNF418", "ZNF419", "ZNF441",
    "ZNF443", "ZNF540", "ZNF563", "ZNF566", "ZNF580", "ZNF595", "ZNF605",
    "ZNF653", "ZNF695", "ZNF764", "ZNF793", "ZNF823"
]

In [5]:
firefate_dynamic_tfs = ["CREB3L2", "IRF2", "IRF7", "MEF2C", "IKZF3", "CDC5L", "POU2F1",
"MAX", "MEF2A", "IRF1", "USF2", "TFEC", "TEAD2", "IRF8"]

TFs_dynamic_scores = all[all["goi"].isin(firefate_dynamic_tfs)]
print(TFs_dynamic_scores.shape)
TFs_dictys_scores = all[all["goi"].isin(dictys_tfs)]
print(TFs_dictys_scores.shape)

(6, 16)
(19, 16)


In [6]:
# Filter dataframe
subset_df = all[~all["goi"].isin(firefate_dynamic_tfs)]

firefate_dynamic_df = all[all["goi"].isin(firefate_dynamic_tfs)]

# Optional: check result
print(subset_df.shape)
print(firefate_dynamic_df.shape)

(101, 16)
(6, 16)


In [7]:
allexceptdynamic_df = all[~all["goi"].isin(TFs_dynamic_scores['goi'])]
print(allexceptdynamic_df.shape)

u_stat, p_value = stats.mannwhitneyu(
    TFs_dynamic_scores["overall_total"],
    allexceptdynamic_df["overall_total"],
    #alternative="two-sided"
    alternative="greater"
)

print("Mann–Whitney U p-value for dynamic:", p_value)

(101, 16)
Mann–Whitney U p-value for dynamic: 0.00797764997988746


In [8]:
# --- FIREFate dynamic vs. random background, and vs. Dictys ---

ff_vals = all[all["goi"].isin(firefate_dynamic_tfs)]["overall_total"].values

# Random / background: all TFs except FIREFate dynamic
other_vals = all[~all["goi"].isin(firefate_dynamic_tfs)]["overall_total"].values

# Dictys hits, excluding the overlap with FIREFate dynamic (IRF8)
dictys_only_vals = all[
    all["goi"].isin(dictys_tfs) & ~all["goi"].isin(firefate_dynamic_tfs)
]["overall_total"].values

u_stat, p_value = stats.mannwhitneyu(ff_vals, other_vals, alternative="greater")
print(f"FIREFate dynamic (n={len(ff_vals)}) vs all other TFs (n={len(other_vals)}): "
      f"U={u_stat:.1f}, p={p_value:.3e}")

u_stat, p_value = stats.mannwhitneyu(ff_vals, dictys_only_vals, alternative="greater")
print(f"FIREFate dynamic (n={len(ff_vals)}) vs Dictys TFs (n={len(dictys_only_vals)}): "
      f"U={u_stat:.1f}, p={p_value:.3e}")

FIREFate dynamic (n=6) vs all other TFs (n=101): U=477.0, p=7.978e-03
FIREFate dynamic (n=6) vs Dictys TFs (n=18): U=73.0, p=1.120e-01


In [ ]:
fig = plt.figure(figsize=(5,6))

positions = [1, 1.55, 2.1]
data = [ff_vals, dictys_only_vals, other_vals]
labels = ["FIREFate dynamic", "Dictys", "Other"]

# boxplot
plt.boxplot(data, positions=positions, widths=0.3)

colors = ["tab:orange", "tab:blue", "#bbbbbb"]

# overlay individual TF points
for i, y in enumerate(data):
    x = np.random.normal(positions[i], 0.03, size=len(y))
    plt.scatter(x, y, alpha=0.8, color=colors[i])

plt.xticks(positions, labels)
plt.xlim(0.75, 2.35)
plt.ylabel("Overall perturbation magnitude")

plt.tight_layout()
# save the plot
fig.savefig(
    os.path.join("/projects/bhdw/asachan/papers/firefate/figures",
                 "dictys_benchmark_ff_dynamic_tonsil.pdf"),
    bbox_inches="tight"
)
plt.show()